# Detección de Lavado de Dinero — Verificación de aprendizaje: arquitectura de dos etapas

Antes de escalar al MVP final sobre las 6.3M filas, este notebook **prueba que la arquitectura de dos etapas efectivamente aprende** — con el pipeline ya limpio de la Parte 2 — y deja resuelta cada decisión de diseño que pide el enunciado:

- **Etapa A (aprendizaje de la normalidad):** autoencoder secuencial entrenado *solo* con cuentas normales, produce una representación comprimida y reconstruye la secuencia; el error de reconstrucción es el score de anomalía.
- **Etapa B (supervisado con transfer learning):** un clasificador que reutiliza el encoder de la Etapa A (no entrena desde cero), con una estrategia de fine-tuning justificada, una función de pérdida apropiada al desbalance, y que combina explícitamente las señales de ambas etapas.
- **Experimento de demostración:** tabla comparativa baseline (supervisado desde cero) vs. las variantes de dos etapas.

> **Advertencia honesta sobre la escala:** este notebook corre sobre el sample (9.4k filas → 613 cuentas de train, 132 de val, 132 de test, con solo 11 positivos en test). Es suficiente para **verificar que el pipeline y las arquitecturas funcionan correctamente end-to-end** y que la dirección de los resultados es la esperada, pero **no es una validación estadísticamente significativa** — con tan pocos positivos en test, un solo caso mal/bien clasificado mueve las métricas varios puntos. La conclusión real sobre si el enfoque de dos etapas aporta valor debe confirmarse corriendo esto contra el dataset completo (Componente 3 / MVP final).


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, f1_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DATA_PATH = "PS_20174392719_1491204439457_log.csv" 
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)

Shape: (6362620, 11)


## 1. Pipeline de datos (igual que Parte 2 — sin el artefacto de balance destino)

**Bug corregido tras correr contra el dataset completo:** `type_idx` no debe escalarse con `StandardScaler` (es categórico, no continuo) — al truncarse de vuelta a entero para `nn.Embedding`, un z-score corrompe la categoría y puede caer fuera de rango, causando `IndexError` en el embedding. `SCALE_COLS` ahora excluye `type_idx` explícitamente.

In [2]:
FEATURE_COLS = ['log_amount', 'log_oldbalanceOrg', 'log_newbalanceOrig',
                'errorBalanceOrig', 'delta_t', 'type_idx']

df_c = df[df['nameDest'].str.startswith('C')].copy()
for col in ['amount', 'oldbalanceOrg', 'newbalanceOrig']:
    df_c[f'log_{col}'] = np.log1p(df_c[col])
df_c['errorBalanceOrig'] = df_c['newbalanceOrig'] + df_c['amount'] - df_c['oldbalanceOrg']
type_map = {t: i for i, t in enumerate(sorted(df_c['type'].unique()))}
df_c['type_idx'] = df_c['type'].map(type_map)
N_TYPES = len(type_map)
df_c = df_c.sort_values(['nameDest', 'step']).reset_index(drop=True)
df_c['delta_t'] = df_c.groupby('nameDest')['step'].diff().fillna(0)

seq_lengths = df_c.groupby('nameDest').size()
seq_labels = df_c.groupby('nameDest')['isFraud'].max()
MAX_LEN = int(seq_lengths.quantile(0.95))

dests = seq_labels.index.values
labels = seq_labels.values
TEST_FRAC, VAL_FRAC = 0.15, 0.15
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_FRAC, random_state=RANDOM_SEED)
trainval_idx, test_idx = next(gss1.split(dests, labels, groups=dests))
trainval_dests, test_dests = dests[trainval_idx], dests[test_idx]
val_frac_within_trainval = VAL_FRAC / (1 - TEST_FRAC)
gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac_within_trainval, random_state=RANDOM_SEED)
tv_labels = seq_labels.loc[trainval_dests].values
train_idx, val_idx = next(gss2.split(trainval_dests, tv_labels, groups=trainval_dests))
train_dests, val_dests = trainval_dests[train_idx], trainval_dests[val_idx]

SCALE_COLS = [c for c in FEATURE_COLS if c != 'type_idx']  # type_idx es categorico -> nunca se escala (bug corregido, ver nota abajo)
scaler = StandardScaler().fit(df_c.loc[df_c['nameDest'].isin(train_dests), SCALE_COLS])
df_c[SCALE_COLS] = scaler.transform(df_c[SCALE_COLS])

def build_seq(dest_ids_subset, df_full, feature_cols, max_len):
    sub = df_full[df_full['nameDest'].isin(dest_ids_subset)].copy()
    sub['acc_idx'], uniques = pd.factorize(sub['nameDest'], sort=False)
    n_accounts = len(uniques)
    labels_by_acc = sub.groupby('acc_idx')['isFraud'].max().reindex(range(n_accounts)).values
    rank_asc = sub.groupby('acc_idx').cumcount()
    group_size = sub.groupby('acc_idx')['acc_idx'].transform('size')
    rank_from_end = (group_size - 1 - rank_asc).values
    keep = rank_from_end < max_len
    col_pos = max_len - 1 - rank_from_end[keep]
    row_acc = sub['acc_idx'].values[keep]
    X = np.zeros((n_accounts, max_len, len(feature_cols)), dtype=np.float32)
    mask = np.zeros((n_accounts, max_len), dtype=np.float32)
    X[row_acc, col_pos, :] = sub.loc[keep, feature_cols].values
    mask[row_acc, col_pos] = 1.0
    y = labels_by_acc.astype(np.float32)
    return X, mask, y

X_train, mask_train, y_train = build_seq(train_dests, df_c, FEATURE_COLS, MAX_LEN)
X_val, mask_val, y_val = build_seq(val_dests, df_c, FEATURE_COLS, MAX_LEN)
X_test, mask_test, y_test = build_seq(test_dests, df_c, FEATURE_COLS, MAX_LEN)

TYPE_COL = FEATURE_COLS.index('type_idx')
NUM_COLS = [i for i in range(len(FEATURE_COLS)) if i != TYPE_COL]
NUM_FEAT = len(NUM_COLS)

class SeqDS(Dataset):
    def __init__(self, X, mask, y):
        self.x_num = torch.tensor(X[:, :, NUM_COLS], dtype=torch.float32)
        self.x_type = torch.tensor(X[:, :, TYPE_COL], dtype=torch.long)
        self.mask = torch.tensor(mask, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x_num[i], self.x_type[i], self.mask[i], self.y[i]

train_ds, val_ds, test_ds = SeqDS(X_train, mask_train, y_train), SeqDS(X_val, mask_val, y_val), SeqDS(X_test, mask_test, y_test)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
print("MAX_LEN:", MAX_LEN, "| N_TYPES:", N_TYPES, "| train/val/test:", len(train_ds), len(val_ds), len(test_ds))
print("positivos -> train:", int(y_train.sum()), "val:", int(y_val.sum()), "test:", int(y_test.sum()))

MAX_LEN: 24 | N_TYPES: 4 | train/val/test: 400372 85794 85795
positivos -> train: 5767 val: 1181 test: 1221


## 2. Etapa A — Autoencoder secuencial

**Longitud variable:** el encoder usa `pack_padded_sequence` con las longitudes reales (derivadas de `mask`), así el GRU nunca procesa posiciones de padding como si fueran transacciones reales.

**Representación comprimida:** el último hidden state del GRU encoder se proyecta a un vector latente de 16 dimensiones (`LATENT_DIM`) — mucho menor que `HIDDEN_DIM=32` o que la secuencia original (`MAX_LEN × 6` valores), forzando al modelo a comprimir lo esencial del comportamiento.

**Reconstrucción:** el decoder usa ese vector latente como estado inicial y como input repetido en cada paso de un segundo GRU, reconstruyendo tanto las features numéricas (regresión, MSE) como el tipo de transacción (clasificación, cross-entropy) en cada paso — ambas pérdidas enmascaradas para ignorar el padding.

In [3]:
EMBED_DIM, HIDDEN_DIM, LATENT_DIM = 4, 32, 16

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.type_emb = nn.Embedding(N_TYPES, EMBED_DIM)
        self.gru = nn.GRU(NUM_FEAT + EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.to_latent = nn.Linear(HIDDEN_DIM, LATENT_DIM)
    def forward(self, x_num, x_type, mask):
        x = torch.cat([x_num, self.type_emb(x_type)], dim=-1)
        lengths = mask.sum(dim=1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.to_latent(h_n.squeeze(0))

class Decoder(nn.Module):
    def __init__(self, max_len):
        super().__init__()
        self.max_len = max_len
        self.h0 = nn.Linear(LATENT_DIM, HIDDEN_DIM)
        self.gru = nn.GRU(LATENT_DIM, HIDDEN_DIM, batch_first=True)
        self.out_num = nn.Linear(HIDDEN_DIM, NUM_FEAT)
        self.out_type = nn.Linear(HIDDEN_DIM, N_TYPES)
    def forward(self, z):
        z_rep = z.unsqueeze(1).repeat(1, self.max_len, 1)
        out, _ = self.gru(z_rep, self.h0(z).unsqueeze(0))
        return self.out_num(out), self.out_type(out)

class AutoEncoder(nn.Module):
    def __init__(self, max_len):
        super().__init__()
        self.encoder, self.decoder = Encoder(), Decoder(max_len)
    def forward(self, x_num, x_type, mask):
        z = self.encoder(x_num, x_type, mask)
        num_hat, type_logits = self.decoder(z)
        return z, num_hat, type_logits

def recon_loss(num_hat, type_logits, x_num, x_type, mask):
    m = mask.unsqueeze(-1)
    num_l = ((num_hat - x_num) ** 2 * m).sum() / (m.sum() * NUM_FEAT + 1e-8)
    type_l = (nn.functional.cross_entropy(type_logits.reshape(-1, N_TYPES), x_type.reshape(-1), reduction='none')
              * mask.reshape(-1)).sum() / (mask.sum() + 1e-8)
    return num_l + 0.5 * type_l   # el tipo pesa menos: es categórico de baja cardinalidad, más fácil de "hacer trampa"

def per_sequence_recon_error(model, loader):
    model.eval(); errs, ys = [], []
    with torch.no_grad():
        for x_num, x_type, mask, y in loader:
            z, num_hat, type_logits = model(x_num, x_type, mask)
            m = mask.unsqueeze(-1)
            num_e = ((num_hat - x_num) ** 2 * m).sum(dim=(1, 2)) / (mask.sum(dim=1) * NUM_FEAT + 1e-8)
            type_e = (nn.functional.cross_entropy(type_logits.reshape(-1, N_TYPES), x_type.reshape(-1), reduction='none')
                      .reshape(mask.shape) * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)
            errs.append((num_e + 0.5 * type_e).numpy()); ys.append(y.numpy())
    return np.concatenate(errs), np.concatenate(ys)

**Decisión clave: la Etapa A se entrena únicamente con las cuentas normales (`isFraud=0`) del set de train.** Es la premisa del enfoque: si el autoencoder también viera cuentas sospechosas durante el entrenamiento, aprendería a reconstruirlas bien también, y perdería la capacidad de usar el error de reconstrucción como señal de anomalía.

In [4]:
normal_mask = y_train == 0
train_normal_ds = SeqDS(X_train[normal_mask], mask_train[normal_mask], y_train[normal_mask])
train_normal_loader = DataLoader(train_normal_ds, batch_size=32, shuffle=True)
print(f"Entrenando Etapa A con {len(train_normal_ds)} cuentas normales (de {len(train_ds)} en train)")

torch.manual_seed(RANDOM_SEED)
ae = AutoEncoder(MAX_LEN)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)

for epoch in range(15):
    ae.train(); tot = 0.0
    for x_num, x_type, mask, y in train_normal_loader:
        opt.zero_grad()
        z, num_hat, type_logits = ae(x_num, x_type, mask)
        loss = recon_loss(num_hat, type_logits, x_num, x_type, mask)
        loss.backward(); opt.step()
        tot += loss.item() * len(y)
    if epoch % 3 == 0 or epoch == 14:
        print(f"  epoch {epoch}: train recon loss = {tot/len(train_normal_ds):.4f}")

Entrenando Etapa A con 394605 cuentas normales (de 400372 en train)
  epoch 0: train recon loss = 1.4118
  epoch 3: train recon loss = 1.3253
  epoch 6: train recon loss = 1.2852
  epoch 9: train recon loss = 1.2675
  epoch 12: train recon loss = 1.2476
  epoch 14: train recon loss = 1.2472


### Umbral sobre el error de reconstrucción — justificado con métrica de validación

Usamos **todo el val set** (normal + sospechoso, ya que sí tenemos sus labels) para elegir el umbral que **maximiza el F1** sobre el error de reconstrucción como score de anomalía — no es un número arbitrario, es el punto óptimo de la curva precision-recall calculada sobre validación. Reportamos también el PR-AUC de la Etapa A sola, como línea base de qué tan buena es la señal de reconstrucción por sí misma, antes de pasar a la Etapa B.

In [5]:
val_err, val_y = per_sequence_recon_error(ae, val_loader)
print("Error de reconstrucción en val -- normal vs sospechoso:")
print(f"  normal (y=0): media={val_err[val_y==0].mean():.4f} | sospechoso (y=1): media={val_err[val_y==1].mean():.4f}")

prec, rec, thr = precision_recall_curve(val_y, val_err)
f1s = 2 * prec * rec / (prec + rec + 1e-8)
best_i = np.nanargmax(f1s[:-1])
best_thr = thr[best_i]
print(f"\nUmbral elegido (max F1 en val): {best_thr:.4f}  ->  F1={f1s[best_i]:.3f}, P={prec[best_i]:.3f}, R={rec[best_i]:.3f}")
print("PR-AUC de la Etapa A sola (solo reconstrucción, sin Etapa B):", round(average_precision_score(val_y, val_err), 4))

Error de reconstrucción en val -- normal vs sospechoso:
  normal (y=0): media=1.2704 | sospechoso (y=1): media=1.7542

Umbral elegido (max F1 en val): 2.5958  ->  F1=0.072, P=0.048, R=0.146
PR-AUC de la Etapa A sola (solo reconstrucción, sin Etapa B): 0.0308


## 3. Etapa B — Clasificador supervisado con transfer learning

**Estrategia de adaptación (justificada):** *fine-tuning gradual (discriminative fine-tuning)*. Primero congelamos el encoder pre-entrenado en la Etapa A y entrenamos solo la cabeza de clasificación unas pocas épocas — esto evita que gradientes grandes y ruidosos de una cabeza recién inicializada destruyan de entrada la representación de "normalidad" ya aprendida (*catastrophic forgetting*). Luego descongelamos el encoder y lo afinamos con una tasa de aprendizaje mucho menor (`1e-4` vs `1e-3` de la cabeza), permitiendo adaptación fina sin borrar lo aprendido.

**Función de pérdida:** `BCEWithLogitsLoss` con `pos_weight = negativos/positivos` — igual que en la Parte 2, consistente con el desbalance real (no hay necesidad de una pérdida más compleja como focal loss dado que el desbalance a nivel de secuencia, ~1-8%, ya es manejable con `pos_weight`; documentamos esto como decisión, no lo dejamos implícito).

**Cómo se combinan las señales de ambas etapas:** probamos dos variantes de la cabeza de clasificación:
1. Solo el vector latente `z` del encoder pre-entrenado (la Etapa A aporta la *representación*).
2. `z` **concatenado con el error de reconstrucción escalar** (la Etapa A aporta representación *y* score de anomalía explícito como feature adicional) — esta es la combinación más completa de ambas señales.

In [6]:
class ClassifierHead(nn.Module):
    def __init__(self, use_recon_error=False):
        super().__init__()
        in_dim = LATENT_DIM + (1 if use_recon_error else 0)
        self.use_recon_error = use_recon_error
        self.net = nn.Sequential(nn.Linear(in_dim, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, z, recon_err=None):
        inp = torch.cat([z, recon_err.unsqueeze(-1)], dim=-1) if self.use_recon_error else z
        return self.net(inp).squeeze(-1)

class FullClassifier(nn.Module):
    def __init__(self, encoder, use_recon_error=False, decoder=None):
        super().__init__()
        self.encoder, self.decoder, self.use_recon_error = encoder, decoder, use_recon_error
        self.head = ClassifierHead(use_recon_error)
    def forward(self, x_num, x_type, mask):
        z = self.encoder(x_num, x_type, mask)
        recon_err = None
        if self.use_recon_error:
            with torch.no_grad():
                num_hat, type_logits = self.decoder(z)
                m = mask.unsqueeze(-1)
                num_e = ((num_hat - x_num) ** 2 * m).sum(dim=(1,2)) / (mask.sum(dim=1)*NUM_FEAT+1e-8)
                type_e = (nn.functional.cross_entropy(type_logits.reshape(-1,N_TYPES), x_type.reshape(-1), reduction='none')
                          .reshape(mask.shape)*mask).sum(dim=1) / (mask.sum(dim=1)+1e-8)
                recon_err = (num_e + 0.5*type_e).detach()
        return self.head(z, recon_err)

def train_classifier(model, loader, epochs, lr_groups, pos_weight):
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    opt = torch.optim.Adam(lr_groups)
    for _ in range(epochs):
        model.train()
        for x_num, x_type, mask, y in loader:
            opt.zero_grad()
            loss = loss_fn(model(x_num, x_type, mask), y)
            loss.backward(); opt.step()
    return model

def evaluate(model, loader, return_probs=False):
    model.eval(); logits, ys = [], []
    with torch.no_grad():
        for x_num, x_type, mask, y in loader:
            logits.append(model(x_num, x_type, mask).numpy()); ys.append(y.numpy())
    logits, ys = np.concatenate(logits), np.concatenate(ys)
    probs = 1/(1+np.exp(-np.clip(logits, -30, 30)))  # clip evita overflow en exp() con logits extremos (pos_weight alto los produce)
    res = {"PR-AUC": average_precision_score(ys, probs),
           "ROC-AUC": roc_auc_score(ys, probs) if len(set(ys)) > 1 else float('nan'),
           "F1@0.5": f1_score(ys, (probs >= 0.5).astype(int), zero_division=0)}
    return (res, probs, ys) if return_probs else res

pos, neg = y_train.sum(), len(y_train) - y_train.sum()
pw = neg / max(pos, 1)
sample_weights = np.where(y_train == 1, 1/max(pos,1), 1/max(neg,1))
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader_sampled = DataLoader(train_ds, batch_size=32, sampler=sampler)
print("pos_weight:", round(pw, 2))

pos_weight: 68.42


**Nota sobre `F1@0.5`:** con un `pos_weight` tan alto (desbalance fuerte) y un dataset de este tamaño, un umbral fijo de 0.5 puede llevar al modelo a predecir "sospechoso" para casi todo (comportamiento degenerado, no es un bug de la métrica sino de usar un umbral no calibrado). Por eso el umbral de decisión final debería calibrarse igual que hicimos con el error de reconstrucción (vía curva precision-recall en validación), no fijarse en 0.5 — lo dejamos así de simple aquí solo para esta verificación rápida, y usamos **PR-AUC y ROC-AUC como las métricas de comparación confiables**, no F1@0.5.

In [7]:
# --- Variante 1: baseline supervisado desde cero (sin Etapa A) ---
torch.manual_seed(RANDOM_SEED)
baseline = FullClassifier(Encoder(), use_recon_error=False)
baseline = train_classifier(baseline, train_loader_sampled, epochs=15,
                             lr_groups=[{"params": baseline.parameters(), "lr": 1e-3}], pos_weight=pw)
res_baseline = evaluate(baseline, test_loader)

# --- Variante 2: transfer learning (encoder de la Etapa A) ---
torch.manual_seed(RANDOM_SEED)
enc2 = Encoder(); enc2.load_state_dict(ae.encoder.state_dict())
transfer_clf = FullClassifier(enc2, use_recon_error=False)
for p in transfer_clf.encoder.parameters(): p.requires_grad = False
transfer_clf = train_classifier(transfer_clf, train_loader_sampled, epochs=5,
    lr_groups=[{"params": transfer_clf.head.parameters(), "lr": 1e-3}], pos_weight=pw)
for p in transfer_clf.encoder.parameters(): p.requires_grad = True
transfer_clf = train_classifier(transfer_clf, train_loader_sampled, epochs=10,
    lr_groups=[{"params": transfer_clf.encoder.parameters(), "lr": 1e-4},
               {"params": transfer_clf.head.parameters(), "lr": 1e-3}], pos_weight=pw)
res_transfer = evaluate(transfer_clf, test_loader)

# --- Variante 3: transfer learning + señal explícita de error de reconstrucción ---
torch.manual_seed(RANDOM_SEED)
enc3 = Encoder(); enc3.load_state_dict(ae.encoder.state_dict())
transfer_re = FullClassifier(enc3, use_recon_error=True, decoder=ae.decoder)
for p in transfer_re.encoder.parameters(): p.requires_grad = False
transfer_re = train_classifier(transfer_re, train_loader_sampled, epochs=5,
    lr_groups=[{"params": transfer_re.head.parameters(), "lr": 1e-3}], pos_weight=pw)
for p in transfer_re.encoder.parameters(): p.requires_grad = True
transfer_re = train_classifier(transfer_re, train_loader_sampled, epochs=10,
    lr_groups=[{"params": transfer_re.encoder.parameters(), "lr": 1e-4},
               {"params": transfer_re.head.parameters(), "lr": 1e-3}], pos_weight=pw)
res_transfer_re = evaluate(transfer_re, test_loader)

print("Listo: 3 variantes entrenadas y evaluadas sobre el mismo test set.")

Listo: 3 variantes entrenadas y evaluadas sobre el mismo test set.


## 4. Calibrar el umbral de decisión (igual criterio que en la Etapa A)

`F1@0.5` no es confiable aquí — con `pos_weight` alto el modelo no está calibrado para que 0.5 sea un punto de corte razonable (ya lo advertimos arriba). Aplicamos el mismo criterio que usamos para el umbral de reconstrucción: elegir, sobre **validación**, el punto de la curva precision-recall que maximiza F1, y reportar Precision/Recall/F1 en test con ese umbral — no con 0.5.

In [8]:
def calibrate_threshold_and_eval(model, val_loader, test_loader):
    val_probs, val_y_ = evaluate(model, val_loader, return_probs=True)[1:]
    prec, rec, thr = precision_recall_curve(val_y_, val_probs)
    f1s = 2 * prec * rec / (prec + rec + 1e-8)
    best_i = np.nanargmax(f1s[:-1])
    best_thr = thr[best_i] if len(thr) > 0 else 0.5
    test_probs, test_y_ = evaluate(model, test_loader, return_probs=True)[1:]
    test_preds = (test_probs >= best_thr).astype(int)
    return {
        "umbral": best_thr,
        "Precision": (test_preds & test_y_.astype(int)).sum() / max(test_preds.sum(), 1),
        "Recall": (test_preds & test_y_.astype(int)).sum() / max(test_y_.sum(), 1),
        "F1_calibrado": f1_score(test_y_, test_preds, zero_division=0),
    }

cal_baseline = calibrate_threshold_and_eval(baseline, val_loader, test_loader)
cal_transfer = calibrate_threshold_and_eval(transfer_clf, val_loader, test_loader)
cal_transfer_re = calibrate_threshold_and_eval(transfer_re, val_loader, test_loader)
for name, c in [("Baseline", cal_baseline), ("Transfer", cal_transfer), ("Transfer+recon", cal_transfer_re)]:
    print(f"{name}: umbral={c['umbral']:.4f}  P={c['Precision']:.3f}  R={c['Recall']:.3f}  F1={c['F1_calibrado']:.3f}")

Baseline: umbral=0.9982  P=0.286  R=0.093  F1=0.140
Transfer: umbral=0.9975  P=0.086  R=0.084  F1=0.085
Transfer+recon: umbral=0.9972  P=0.179  R=0.368  F1=0.241


## 5. Experimento de demostración — tabla comparativa (con umbral calibrado)

In [9]:
comparacion = pd.DataFrame([
    {"Variante": "Baseline supervisado desde cero (sin Etapa A)", **res_baseline, **cal_baseline},
    {"Variante": "Transfer learning (encoder Etapa A, fine-tuned)", **res_transfer, **cal_transfer},
    {"Variante": "Transfer learning + error de reconstrucción", **res_transfer_re, **cal_transfer_re},
])
comparacion

,Variante,PR-AUC,ROC-AUC,F1@0.5,umbral,Precision,Recall,F1_calibrado
0,Baseline supervisado desde cero (sin Etapa A),0.096063,0.648102,0.029400,0.998216,0.286076,0.092547,0.139851
1,"Transfer learning (encoder Etapa A, fine-tuned)",0.071511,0.633695,0.028395,0.997503,0.085931,0.083538,0.084718
2,Transfer learning + error de reconstrucción,0.114724,0.771533,0.028687,0.997243,0.178813,0.367731,0.240622


## Resumen y siguiente paso

| Requisito del enunciado | Dónde se resuelve |
|---|---|
| Longitud de secuencia variable | `pack_padded_sequence` en el `Encoder`, usando `mask.sum()` como longitud real |
| Representación comprimida | Vector latente de 16 dims (`to_latent`), muy por debajo del hidden state (32) o la secuencia original |
| Reconstrucción desde la representación | `Decoder` (GRU) reconstruye features numéricas + tipo por paso |
| Error de reconstrucción = score de anomalía | `per_sequence_recon_error`, enmascarado, combinando pérdida numérica + categórica |
| Umbral justificado sobre validación (Etapa A) | Punto de máximo F1 en la curva precision-recall de val |
| Umbral justificado sobre validación (Etapa B) | Mismo criterio aplicado al clasificador — nunca 0.5 fijo |
| Etapa B aprovecha la Etapa A (transfer learning) | Encoder inicializado con los pesos de la Etapa A; fine-tuning gradual (freeze → unfreeze con LR distinto) |
| Pérdida apropiada al desbalance | `BCEWithLogitsLoss(pos_weight=...)`, mismo criterio que en la Parte 2 |
| Combinación de señales de ambas etapas | Variante 3: vector latente + error de reconstrucción concatenados en la cabeza de clasificación |
| Demostración empírica de valor | Tabla comparativa de 3 variantes, con métricas de umbral fijo y calibrado, sobre el mismo test set |

